In [20]:
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

In [ ]:

# SOURCE: https://surfer.nmr.mgh.harvard.edu/fswiki/FsTutorial/AnatomicalROI/FreeSurferColorLUT
# ── FreeSurfer LUT (partial — key ones) ───────────────────────────────────────
FS_LUT = {
    0:  'Unknown',
    1:  'Left-Cerebral-Exterior',
    2:  'Left-Cerebral-White-Matter',
    3:  'Left-Cerebral-Cortex',
    4:  'Left-Lateral-Ventricle',
    5:  'Left-Inf-Lat-Vent',
    6:  'Left-Cerebellum-Exterior',
    7:  'Left-Cerebellum-White-Matter',
    8:  'Left-Cerebellum-Cortex',
    9:  'Left-Thalamus',
    10: 'Left-Thalamus-Proper',
    11: 'Left-Caudate',
    12: 'Left-Putamen',
    13: 'Left-Pallidum',
    14: '3rd-Ventricle',
    15: '4th-Ventricle',
    16: 'Brain-Stem',
    17: 'Left-Hippocampus',
    18: 'Left-Amygdala',
    19: 'Left-Insula',
    20: 'Left-Operculum',
    24: 'CSF',
    25: 'Left-Lesion',
    26: 'Left-Accumbens-area',
    27: 'Left-Substancia-Nigra',
    28: 'Left-VentralDC',
    29: 'Left-undetermined',
    30: 'Left-vessel',
    31: 'Left-choroid-plexus',
    40: 'Right-Cerebral-Exterior',
    41: 'Right-Cerebral-White-Matter',
    42: 'Right-Cerebral-Cortex',
    43: 'Right-Lateral-Ventricle',
    44: 'Right-Inf-Lat-Vent',
    45: 'Right-Cerebellum-Exterior',
    46: 'Right-Cerebellum-White-Matter',
    47: 'Right-Cerebellum-Cortex',
    48: 'Right-Thalamus',
    49: 'Right-Thalamus-Proper',
    50: 'Right-Caudate',
    51: 'Right-Putamen',
    52: 'Right-Pallidum',
    53: 'Right-Hippocampus',
    54: 'Right-Amygdala',
    55: 'Right-Insula',
    56: 'Right-Operculum',
    57: 'Right-Lesion',
    58: 'Right-Accumbens-area',
    59: 'Right-Substancia-Nigra',
    60: 'Right-VentralDC',
    61: 'Right-undetermined',
    62: 'Right-vessel',
    63: 'Right-choroid-plexus',
    72: '5th-Ventricle',
    73: 'Left-Interior',
    74: 'Right-Interior',
}


In [ ]:

# ── Paths ──────────────────────────────────────────────────────────────────────
LBCN_SUBDIR  = Path( << INSERT >> )
T1_PATH      = LBCN_SUBDIR / 'Freesurfer' / SUB / 'elec_recon' / 'T1.nii.gz'
ASEG_PATH    = LBCN_SUBDIR / 'Freesurfer' / SUB / 'mri' / 'aseg.nii.gz'
COORDS_TSV   = LBCN_SUBDIR / f'{SUB}_MEC_elec_loc.xlsx'   # ScannerNativeRAS columns
OUT_DIR      = Path( << INSERT OUTPUT FOLDER >> )
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load images ────────────────────────────────────────────────────────────────
t1        = nib.load(T1_PATH)
aseg      = nib.load(ASEG_PATH)
t1_data   = t1.get_fdata()
aseg_data = aseg.get_fdata()
affine    = t1.affine          # assumes T1 and aseg share affine
inv_aff   = np.linalg.inv(affine)
print(f"our T1 orientiation is : {nib.aff2axcodes(t1.affine)}")

our T1 orientiation is : ('L', 'I', 'A')


In [ ]:


# ── Load electrode tables ──────────────────────────────────────────────────────
coords_df = pd.read_excel(COORDS_TSV) 
elec_df   = coords_df[['chan_num', 
                       'FS_label', 
                       'ScannerNativeRAS_coord_1',	
                       'ScannerNativeRAS_coord_2',	
                       'ScannerNativeRAS_coord_3']] #.join(labels_df, how='left')     # aligned by chan_num

# ── Helper: mm (RAS) → voxel ──────────────────────────────────────────────────
def ras_to_vox(xyz, inv_aff):
    c = np.array([*xyz, 1.0])
    return np.round(inv_aff @ c).astype(int)[:3]

# ── Lookup aseg label for every electrode ─────────────────────────────────────
aseg_labels = []
for _, row in elec_df.iterrows():
    vox = ras_to_vox([row.ScannerNativeRAS_coord_1, 
                      row.ScannerNativeRAS_coord_2, 
                      row.ScannerNativeRAS_coord_3], inv_aff)
    # clamp to volume bounds
    vox = np.clip(vox, 0, np.array(aseg_data.shape) - 1)
    label_id   = int(aseg_data[vox[0], vox[1], vox[2]])
    label_name = FS_LUT.get(label_id, f'Unknown({label_id})')
    aseg_labels.append((label_id, label_name))

elec_df['aseg_id']   = [a[0] for a in aseg_labels]
elec_df['aseg_label'] = [a[1] for a in aseg_labels]

print(elec_df[['FS_label', 'aseg_id', 'aseg_label']].to_string())

# ── Iterative snapshot ─────────────────────────────────────────────────────────
for _, row in elec_df.iterrows():
    vox = ras_to_vox([row.ScannerNativeRAS_coord_1, 
                      row.ScannerNativeRAS_coord_2, 
                      row.ScannerNativeRAS_coord_3], inv_aff)
    vox = np.clip(vox, 0, np.array(t1_data.shape) - 1)
    xi, yi, zi = vox

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor('black')
    fig.suptitle(
        f'Ch{row.chan_num}  {row.FS_label}  |  aseg: {row.aseg_label}  '
        f'|  RAS: ({row.ScannerNativeRAS_coord_1:.1f},         {row.ScannerNativeRAS_coord_2:.1f},         {row.ScannerNativeRAS_coord_3:.1f})',
        color='white', fontsize=11
    )

    sx, sy, sz = t1_data.shape

    slices = [
    # Sagittal (X slice)
    (np.flipud(t1_data[xi, :, :].T),
        np.flipud(aseg_data[xi, :, :].T),
        (sy-1-yi, sz-1-zi),
        'Sagittal'),

    # Coronal (Y slice)
    (np.flipud(t1_data[:, yi, :].T),
        np.flipud(aseg_data[:, yi, :].T),
        (xi, sz-1-zi),
        'Coronal'),

    # Axial (Z slice)
    (np.flipud(t1_data[:, :, zi].T),
        np.flipud(aseg_data[:, :, zi].T),
        (xi, sy-1-yi),
        'Axial'),
    ]

    for ax, (t1_sl, aseg_sl, (hx, vy), title) in zip(axes, slices):
        ax.set_facecolor('black')

        # T1 background
        ax.imshow(t1_sl, origin='lower', cmap='gray',
                  vmin=np.percentile(t1_sl, 1),
                  vmax=np.percentile(t1_sl, 99))

        # aseg overlay
        aseg_masked = np.ma.masked_where(aseg_sl == 0, aseg_sl)
        ax.imshow(aseg_masked, origin='lower', cmap='tab20',
                  alpha=0.35, vmin=0, vmax=90)

        # electrode marker
        # ax.scatter(hx, vy, s=30, color='red',
                #    marker='+', linewidths=1, zorder=5)
        ax.scatter(hx, vy, s=5, facecolors='none',
                   edgecolors='red', linewidths=2, zorder=5)

        ax.set_title(title, color='white', fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, f'ch{row.chan_num:03d}_{row.FS_label}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='black')
    plt.close()
    print(f'Saved {out_path}  →  {row.aseg_label}')

print('Done!')

/var/folders/yc/y436cq1j2733vcg9y4l4nrrm0000gp/T/ipykernel_86845/2456059433.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  elec_df['aseg_id']   = [a[0] for a in aseg_labels]
/var/folders/yc/y436cq1j2733vcg9y4l4nrrm0000gp/T/ipykernel_86845/2456059433.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  elec_df['aseg_label'] = [a[1] for a in aseg_labels]


    FS_label  aseg_id                   aseg_label
0      LPIN1        3         Left-Cerebral-Cortex
1      LPIN2        3         Left-Cerebral-Cortex
2      LPIN3        3         Left-Cerebral-Cortex
3      LPIN4        3         Left-Cerebral-Cortex
4      LPIN5        2   Left-Cerebral-White-Matter
5      LPIN6        2   Left-Cerebral-White-Matter
6      LPIN7        3         Left-Cerebral-Cortex
7      LPIN8        2   Left-Cerebral-White-Matter
8      LPIN9        2   Left-Cerebral-White-Matter
9     LPIN10        2   Left-Cerebral-White-Matter
10    LPIN11        2   Left-Cerebral-White-Matter
11    LPIN12        2   Left-Cerebral-White-Matter
12    LPIN13        2   Left-Cerebral-White-Matter
13    LPIN14        3         Left-Cerebral-Cortex
14    LPIN15        3         Left-Cerebral-Cortex
15    LPIN16        2   Left-Cerebral-White-Matter
16     LAIN1        3         Left-Cerebral-Cortex
17     LAIN2        3         Left-Cerebral-Cortex
18     LAIN3        3         L